In [1]:
import os
from pyspark.sql import SparkSession

base_dir = r"E:\BIGDATA\data\parquet"
temp_dir = os.path.join(base_dir, "spark_temp")
os.makedirs(temp_dir, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("cic_clean_write")
    .master("local[*]")
    .config("spark.hadoop.fs.defaultFS", "file:///")
    .config("spark.local.dir", temp_dir)
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.driver.memory", "8g")
    .getOrCreate()
)
spark

In [2]:
parquet_path = r"E:\BIGDATA\data\parquet\cic_ddos_2019_10gb_cleaned.parquet"

df_raw = spark.read.parquet(parquet_path)
df_raw.printSchema()

root
 |-- label: string (nullable = true)
 |-- source_port: integer (nullable = true)
 |-- destination_port: integer (nullable = true)
 |-- protocol: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- flow_duration: long (nullable = true)
 |-- total_fwd_packets: integer (nullable = true)
 |-- total_backward_packets: integer (nullable = true)
 |-- total_length_of_fwd_packets: double (nullable = true)
 |-- total_length_of_bwd_packets: double (nullable = true)
 |-- fwd_packet_length_max: double (nullable = true)
 |-- fwd_packet_length_min: double (nullable = true)
 |-- fwd_packet_length_mean: double (nullable = true)
 |-- fwd_packet_length_std: double (nullable = true)
 |-- bwd_packet_length_max: double (nullable = true)
 |-- bwd_packet_length_min: double (nullable = true)
 |-- bwd_packet_length_mean: double (nullable = true)
 |-- bwd_packet_length_std: double (nullable = true)
 |-- flow_bytes_per_s: double (nullable = true)
 |-- flow_packets_per_s: double (nullabl

In [3]:
from pyspark.sql import functions as F


df = df_raw


print("Rows:", df.count())
df.show(5, truncate=False)

Rows: 24207109
+----------+-----------+----------------+--------+--------------------------+-------------+-----------------+----------------------+---------------------------+---------------------------+---------------------+---------------------+----------------------+---------------------+---------------------+---------------------+----------------------+---------------------+----------------+------------------+-------------+------------+------------+------------+-------------+------------+-----------+-----------+-----------+-------------+------------+-----------+-----------+-----------+-------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+------------------+-----------------+----------------------+--------------+--------------+--------------+--------------+--------------+-----------------+-------------------+--------------------+--------------------+-------------------+-------------------+-----------------+---------

In [4]:
from pyspark.sql import functions as F

label_col = "label"

all_cols = df.columns
numeric_cols = [c for c, t in df.dtypes if c != label_col and t in ("int", "bigint", "double", "float", "long", "smallint", "tinyint")]

len(numeric_cols), numeric_cols[:10]


(69,
 ['source_port',
  'destination_port',
  'protocol',
  'flow_duration',
  'total_fwd_packets',
  'total_backward_packets',
  'total_length_of_fwd_packets',
  'total_length_of_bwd_packets',
  'fwd_packet_length_max',
  'fwd_packet_length_min'])

In [5]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [c for c in numeric_cols if c in df.columns]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

df_feat = assembler.transform(df).select("label", "features")
df_feat.show(5, truncate=False)


+----------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|label     |features                                                                                                                                                                                                                                                                                         |
+----------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|DrDoS_SNMP|(69,[0,1,2,3,4,6,8,9,10,16,17,18,20,21,22,23,25,26,33,35,37,38,39,48,49,51,52,5

In [ ]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

label_indexer = StringIndexer(inputCol="label", outputCol="label_idx", handleInvalid="skip")

rf = RandomForestClassifier(
    labelCol="label_idx",
    featuresCol="features",
    numTrees=300,
    maxDepth=10,
    maxBins=64,
    subsamplingRate=0.4,
    featureSubsetStrategy="sqrt",
    minInstancesPerNode=10,
    seed=42
)


pipe = Pipeline(stages=[label_indexer, rf])
model = pipe.fit(df_feat)

rf_model = model.stages[-1]
importances = rf_model.featureImportances.toArray().tolist()

imp_df = spark.createDataFrame(
    list(zip(feature_cols, importances)),
    ["feature", "importance"]
).orderBy(F.desc("importance"))



In [18]:
feature = sorted(
    zip(feature_cols, importances),
    key=lambda x: x[1],
    reverse=True
)

feature



[('packet_length_mean', 0.10126470454099849),
 ('min_packet_length', 0.07953696723720106),
 ('source_port', 0.07782601707739302),
 ('fwd_packet_length_mean', 0.07400906677942266),
 ('avg_fwd_segment_size', 0.06427685491066176),
 ('average_packet_size', 0.062491333184157366),
 ('fwd_packet_length_min', 0.060567905797086194),
 ('subflow_fwd_bytes', 0.05700909149485593),
 ('flow_bytes_per_s', 0.053528617620394105),
 ('fwd_packet_length_max', 0.047270704582783894),
 ('total_length_of_fwd_packets', 0.03310590457481349),
 ('max_packet_length', 0.029159170685528736),
 ('init_win_bytes_forward', 0.023284838846139813),
 ('ack_flag_count', 0.02146768494894684),
 ('flow_packets_per_s', 0.019674632540018562),
 ('total_fwd_packets', 0.01956112058681088),
 ('flow_iat_min', 0.016140517959478188),
 ('flow_duration', 0.015032561196255288),
 ('act_data_pkt_fwd', 0.012898763902799369),
 ('fwd_iat_max', 0.012549311541420548),
 ('protocol', 0.011084407517047316),
 ('fwd_iat_min', 0.0100888870548341),
 ('fw

In [19]:
pairs = sorted(feature, key=lambda x: x[1], reverse=True)
total = sum(i for _, i in pairs)

selected = []
cum = 0.0
threshold = 0.95

for f, imp in pairs:
    if imp <= 0:
        continue
    selected.append(f)
    cum += imp / total if total else 0.0
    if cum >= threshold:
        break

selected, len(selected), cum

(['packet_length_mean',
  'min_packet_length',
  'source_port',
  'fwd_packet_length_mean',
  'avg_fwd_segment_size',
  'average_packet_size',
  'fwd_packet_length_min',
  'subflow_fwd_bytes',
  'flow_bytes_per_s',
  'fwd_packet_length_max',
  'total_length_of_fwd_packets',
  'max_packet_length',
  'init_win_bytes_forward',
  'ack_flag_count',
  'flow_packets_per_s',
  'total_fwd_packets',
  'flow_iat_min',
  'flow_duration',
  'act_data_pkt_fwd',
  'fwd_iat_max',
  'protocol',
  'fwd_iat_min',
  'fwd_iat_std',
  'flow_iat_max',
  'flow_iat_mean',
  'subflow_fwd_packets',
  'fwd_iat_mean'],
 27,
 0.9513829358800354)

In [20]:
drop_always = {"timestamp", "flow_id", "unnamed_0"}

selected = [c for c in selected if c not in drop_always]

keep_ports = True
if not keep_ports:
    selected = [c for c in selected if c not in {"source_port", "destination_port"}]

selected


['packet_length_mean',
 'min_packet_length',
 'source_port',
 'fwd_packet_length_mean',
 'avg_fwd_segment_size',
 'average_packet_size',
 'fwd_packet_length_min',
 'subflow_fwd_bytes',
 'flow_bytes_per_s',
 'fwd_packet_length_max',
 'total_length_of_fwd_packets',
 'max_packet_length',
 'init_win_bytes_forward',
 'ack_flag_count',
 'flow_packets_per_s',
 'total_fwd_packets',
 'flow_iat_min',
 'flow_duration',
 'act_data_pkt_fwd',
 'fwd_iat_max',
 'protocol',
 'fwd_iat_min',
 'fwd_iat_std',
 'flow_iat_max',
 'flow_iat_mean',
 'subflow_fwd_packets',
 'fwd_iat_mean']

In [22]:
len(selected)

27

In [24]:
from pyspark.sql import functions as F

label_col = "label"

use_cols = [c for c in selected if c in df.columns]
use_cols = use_cols + [label_col]

df_feature = df.select(*use_cols)

df_feature.printSchema()


root
 |-- packet_length_mean: double (nullable = true)
 |-- min_packet_length: double (nullable = true)
 |-- source_port: integer (nullable = true)
 |-- fwd_packet_length_mean: double (nullable = true)
 |-- avg_fwd_segment_size: double (nullable = true)
 |-- average_packet_size: double (nullable = true)
 |-- fwd_packet_length_min: double (nullable = true)
 |-- subflow_fwd_bytes: integer (nullable = true)
 |-- flow_bytes_per_s: double (nullable = true)
 |-- fwd_packet_length_max: double (nullable = true)
 |-- total_length_of_fwd_packets: double (nullable = true)
 |-- max_packet_length: double (nullable = true)
 |-- init_win_bytes_forward: integer (nullable = true)
 |-- ack_flag_count: integer (nullable = true)
 |-- flow_packets_per_s: double (nullable = true)
 |-- total_fwd_packets: integer (nullable = true)
 |-- flow_iat_min: double (nullable = true)
 |-- flow_duration: long (nullable = true)
 |-- act_data_pkt_fwd: integer (nullable = true)
 |-- fwd_iat_max: double (nullable = true)
 |

In [25]:
df_feature.show(5, truncate=False)

+------------------+-----------------+-----------+----------------------+--------------------+-------------------+---------------------+-----------------+----------------+---------------------+---------------------------+-----------------+----------------------+--------------+------------------+-----------------+------------+-------------+----------------+-----------+--------+-----------+-----------+------------+-------------+-------------------+------------+----------+
|packet_length_mean|min_packet_length|source_port|fwd_packet_length_mean|avg_fwd_segment_size|average_packet_size|fwd_packet_length_min|subflow_fwd_bytes|flow_bytes_per_s|fwd_packet_length_max|total_length_of_fwd_packets|max_packet_length|init_win_bytes_forward|ack_flag_count|flow_packets_per_s|total_fwd_packets|flow_iat_min|flow_duration|act_data_pkt_fwd|fwd_iat_max|protocol|fwd_iat_min|fwd_iat_std|flow_iat_max|flow_iat_mean|subflow_fwd_packets|fwd_iat_mean|label     |
+------------------+-----------------+-----------+

In [27]:
output_path = "file:///E:/BIGDATA/data/parquet/cic_ddos_2019_27_features_raw.parquet"

df_feature.write.mode("overwrite").parquet(output_path)


In [28]:
spark.stop()